In [ ]:
!pip install ultralytics pandas matplotlib seaborn -q

print("✅ Packages installed successfully!")

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os

sns.set_style("whitegrid")
print("✅ Libraries imported")

In [ ]:
# Load segmentation model
model = YOLO("yolo11s-seg.pt")   # Small & fast. You can use 'yolo11m-seg.pt' or 'yolo11l-seg.pt' for better accuracy

print("✅ YOLO11 Segmentation model loaded!")

In [ ]:
def analyze_instance_segmentation(image_path):
    """Full analysis: Detection + Segmentation + Area + Color"""
    
    # Load image
    if image_path.startswith("http"):
        import requests
        resp = requests.get(image_path)
        img = cv2.imdecode(np.frombuffer(resp.content, np.uint8), 1)
    else:
        img = cv2.imread(image_path)
    
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Run segmentation
    results = model(rgb_img, conf=0.4, verbose=False)
    
    print(f"🎯 Detected {len(results[0].boxes)} objects\n")
    
    data = []
    
    for i, (box, mask) in enumerate(zip(results[0].boxes, results[0].masks.data)):
        cls_name = results[0].names[int(box.cls)]
        confidence = float(box.conf)
        
        # Convert mask to numpy
        mask_np = mask.cpu().numpy()
        
        # Calculate area (number of pixels)
        area_pixels = np.sum(mask_np > 0.5)
        
        # Approximate real area (very rough - needs calibration)
        approx_area = area_pixels * 0.01  # placeholder scaling factor
        
        # Get color inside the mask
        masked_img = rgb_img.copy()
        masked_img[~mask_np.astype(bool)] = 0
        mean_color = np.mean(masked_img[mask_np.astype(bool)], axis=0).astype(int)
        
        data.append({
            "Object": i+1,
            "Class": cls_name,
            "Confidence": round(confidence, 3),
            "Area (pixels)": area_pixels,
            "Approx Area": round(approx_area, 2),
            "Mean Color (R,G,B)": tuple(mean_color)
        })
        
        print(f"Object {i+1}: {cls_name} | Area: {area_pixels} pixels | Color: {tuple(mean_color)}")
    
    # Create DataFrame
    df = pd.DataFrame(data)
    
    # Visualization
    plt.figure(figsize=(20, 8))
    
    plt.subplot(1, 3, 1)
    plt.imshow(rgb_img)
    plt.title("Original Image")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    annotated = results[0].plot()
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("Instance Segmentation")
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    if len(results[0].masks) > 0:
        combined_mask = np.zeros_like(results[0].masks.data[0].cpu().numpy())
        for mask in results[0].masks.data:
            combined_mask += mask.cpu().numpy()
        plt.imshow(combined_mask, cmap='jet')
        plt.title("All Masks Combined")
        plt.colorbar()
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    return df, results

In [ ]:
# Test 
df1, res1 = analyze_instance_segmentation("https://ultralytics.com/images/bus.jpg")


In [ ]:
# After running analysis
print("📊 Detailed Analysis Report")
print(df1)

# Plot area distribution
plt.figure(figsize=(10, 6))
sns.barplot(data=df1, x="Class", y="Area (pixels)")
plt.title("Object Area Distribution")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Cell: Thermal Image Object Detection

from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt

# Load model (YOLO works decently on thermal, better with fine-tuned models)
model = YOLO("yolo11s.pt")

def detect_thermal(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print("Error loading image")
        return
    
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    results = model(rgb, conf=0.4, verbose=False)
    
    print(f"🔥 Detected {len(results[0].boxes)} objects in thermal image\n")
    
    for box in results[0].boxes:
        cls = results[0].names[int(box.cls)]
        conf = float(box.conf)
        print(f"→ {cls} | Confidence: {conf:.3f}")
    
    # Show result
    annotated = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    plt.title("Thermal Image Object Detection")
    plt.axis('off')
    plt.show()

# Test with a thermal image (you can upload your own)
detect_thermal("thermal_test.jpg")   # Replace with your thermal image path

In [ ]:
# Cell: Multi-Camera Tracking Demo

from ultralytics import YOLO
import cv2

model = YOLO("yolo11s.pt")

def multi_camera_demo(video1_path, video2_path):
    cap1 = cv2.VideoCapture(video1_path)
    cap2 = cv2.VideoCapture(video2_path)
    
    print("🎥 Simulating Multi-Camera Tracking...")
    
    frame_count = 0
    while True:
        ret1, frame1 = cap1.read()
        ret2, frame2 = cap2.read()
        
        if not ret1 or not ret2 or frame_count > 100:   # Limit for demo
            break
        
        # Track on both cameras
        res1 = model.track(frame1, persist=True, tracker="bytetrack.yaml", verbose=False)
        res2 = model.track(frame2, persist=True, tracker="bytetrack.yaml", verbose=False)
        
        # In real system, you would use ReID to match same person across cameras
        annotated1 = res1[0].plot()
        annotated2 = res2[0].plot()
        
        # Show side by side
        combined = np.hstack((annotated1, annotated2))
        cv2.imshow("Camera 1 | Camera 2", combined)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
            
        frame_count += 1
    
    cap1.release()
    cap2.release()
    cv2.destroyAllWindows()

print("✅ Multi-camera tracking function ready")